# 16 - Light Rail, Tram and Minor Modes

The national analysis in notebooks 02-10 is, numerically, an analysis of **buses**: 6,796 of the 7,792 routes and 412,544 of the 420,133 scheduled trips in this feed are `route_type = 3`. Everything else is a rounding error in the aggregate statistics and is therefore invisible in them. This notebook pulls the small modes out of that shadow and studies each one as a network in its own right:

| `route_type` | label used here | what it actually is in this feed |
|---|---|---|
| 0 | tram/light rail | the Jerusalem Red Line and the Tel Aviv Red Line |
| 5 | cable tram | the Haifa Carmelit funicular and the Haifa cable car (Rakavlit) |
| 8 | trolleybus | **not** trolleybuses - shared-taxi (*monit sherut*) operators coded under this type |
| 715 | demand/other bus | rural demand-responsive feeders (Mateh Yehuda, Hevel Eilot, Yoav) |

Rail (`route_type = 2`) is deliberately **not** covered here - notebook 11 already treats it in full.

**Research question addressed here:** *do the minor modes behave like small copies of the bus network, or like something structurally different - and what does that imply for criticality?* The headline tension is service **intensity**: light rail has 8 routes but 2,890 daily trips (~361 trips per route) and cable tram 4 routes with 3,006 trips (~752 trips per route), against ~61 trips per route for bus. Very few stations, very high load per station, and - as the structural section shows - almost no alternative paths. That combination is exactly what makes a station critical, and it is a different failure mode from the one the bus network exhibits.

Because these graphs have tens of nodes rather than tens of thousands, **every centrality here is exact**. Notebook 04 had to approximate betweenness with `k`-sample Brandes on 30k nodes; on a 138-node graph exact Brandes is instantaneous, so there is no reason to accept sampling error and none is accepted.

## Inputs

* `israel-public-transportation/routes.txt`, `trips.txt`, `agency.txt` - mode assignment (`route_type` lives on `routes.txt`; `trips.txt` joins `trip_id` to `route_id`).
* `israel-public-transportation/stop_times.txt` - 816 MB / 15.7M rows, **not tracked in git**, fetched from Google Drive by a cell below and read **streaming, row by row**.
* `outputs/nb/01_data_preparation/tables/stops_clean.csv` - stop names, coordinates, `region`, `metro`. **Required**; produced by notebook `01_data_preparation`.
* `outputs/nb/02_graph_construction/tables/nodes.csv` + `edges.csv` - **optional**; used only for the cross-check that locates the minor modes inside the national graph.
* `outputs/nb/14_*/tables/mode_inventory.csv` - **optional**; if notebook 14 has been run, its inventory is cross-checked against the one computed here.

## Outputs (all under `outputs/nb/16_lightrail_and_minor_modes/`)

| Path | Contents |
|---|---|
| `tables/lightrail_station_metrics.csv` | one row per station of **every** minor mode: `stop_id, stop_name, lat, lon, degree, weighted_degree, mode_label` (+ exact betweenness, calls, articulation flag) |
| `tables/minor_modes_summary.csv` | one row per minor mode: size, shape, redundancy, service intensity, span |
| `lightrail_summary.json` | headline numbers for the report |
| `tables/minor_mode_routes.csv` | every minor-mode route with agency and trip count |
| `tables/mode_service_intensity.csv` | trips per route / calls per stop for **all six** modes, bus and rail included as baseline |
| `tables/minor_mode_components.csv` | every connected component of every minor-mode graph, with its shape class |
| `tables/platform_duplication.csv` | distinct `stop_id` vs distinct `stop_name` per mode (the directional-platform artefact) |
| `tables/minor_mode_single_station_damage.csv` | exact single-station removal damage per mode |
| `tables/hourly_departures.csv` | departures per service hour per mode (hours 24-27 included, see the GTFS time note) |
| `tables/national_component_modes.csv` | which national-graph components the minor modes occupy (only if notebook 02 has been run) |
| `lightrail_graphs.pkl` | `{mode_label: networkx.Graph}` for the four minor modes |
| `figures/*.png` | scale, service intensity, geography, exact betweenness, hourly profile, damage |

Nothing outside this folder is written. In particular `outputs/tables`, `outputs/figures` and `outputs/rail` (the frozen, report-cited outputs) are never touched.

## Notebooks that must run first

* **`01_data_preparation`** - required, for `stops_clean.csv`.
* `02_graph_construction` - optional, only for the national-placement cross-check.
* `14_mode_inventory` - optional, only for a consistency assertion.

This notebook does **not** depend on notebook 15 and does not reuse the bus graph.

## 1. Environment bootstrap

The cell below is the project-standard bootstrap, identical to the one in notebooks 02 and 03. It makes the notebook runnable on a local checkout and on Google Colab: `_ensure(...)` pip-installs only the packages that are genuinely missing (so re-running is cheap), and `find_repo_root()` walks up from the working directory looking for the GTFS folder, cloning the repository into `/content` if we are on Colab and it is not there. It then fixes `REPO`, `DATA` and `OUT`. Every later cell depends on these three paths, so this must run first.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. Libraries, stage folders and cost knobs

We import the scientific stack and fix this stage's folder layout: everything produced here lands in `outputs/nb/16_lightrail_and_minor_modes/` with `tables/` and `figures/` sub-folders, following the one-folder-per-notebook convention.

The constants are gathered here so a grader can see the cost of the notebook in one place:

* `MINOR_ROUTE_TYPES = (0, 5, 8, 715)` - the four modes analysed. Rail (2) belongs to notebook 11 and bus (3) to notebook 15.
* `PROGRESS_EVERY = 2_000_000` - progress prints during the single streaming pass over `stop_times.txt`. **This pass is the only expensive step in the notebook**: 15.7M rows, typically 20-90 seconds depending on disk cache (a few minutes on a cold Colab disk). Everything after it operates on at most a few hundred nodes and is effectively instantaneous.
* `EXACT_BETWEENNESS = True` - use exact Brandes betweenness. Cost is `O(n*m)`: for the largest minor mode here that is roughly `153 * 194 ~ 3e4` operations, i.e. microseconds. Sampling would be pure loss of accuracy for no gain, so the flag exists only to document the choice; leave it on.
* `FIG_DPI`, `TOP_N`, `MAX_LABELLED_STATIONS` - presentation only.

`MODE_LABELS` reuses the exact label strings that notebook 01 wrote into `route_type_distribution.csv`, so tables from different stages join cleanly on `mode_label`.

In [ ]:
# --- Libraries, stage folders and cost knobs ------------------------------
_ensure('pandas', 'numpy', 'networkx', 'matplotlib', 'seaborn')

import csv, json, pickle, time
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.05)
csv.field_size_limit(10_000_000)   # a few rows of stop_times.txt are unusually long

STAGE = OUT / '16_lightrail_and_minor_modes'
TABLES = STAGE / 'tables'
FIGURES = STAGE / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

# --- Cost knobs and mode vocabulary --------------------------------------
MINOR_ROUTE_TYPES = (0, 5, 8, 715)     # tram/light rail, cable tram, trolleybus, demand-responsive
ALL_ROUTE_TYPES = (0, 2, 3, 5, 8, 715)  # used only for the service-intensity baseline
MODE_LABELS = {0: 'tram/light rail', 2: 'rail', 3: 'bus',
               5: 'cable tram', 8: 'trolleybus', 715: 'demand/other bus'}
MODE_SLUGS = {0: 'lightrail', 5: 'cabletram', 8: 'trolleybus', 715: 'demand'}
MODE_COLORS = {0: '#dc2626', 2: '#0f766e', 3: '#94a3b8',
               5: '#0891b2', 8: '#7c3aed', 715: '#ca8a04'}

PROGRESS_EVERY = 2_000_000       # progress print interval for the streaming pass
EXACT_BETWEENNESS = True         # exact Brandes O(n*m) - trivial at this graph size
FIG_DPI = 150                    # figure resolution
TOP_N = 20                       # rows shown in top-N tables
MAX_LABELLED_STATIONS = 80       # safety cap on the labelled light-rail map

print('stage output folder :', STAGE)
print('modes analysed      :', [f'{rt} = {MODE_LABELS[rt]}' for rt in MINOR_ROUTE_TYPES])

## 3. Hebrew label rendering

Every station name in this notebook is Hebrew, and unlike the national maps (30k anonymous dots) the minor-mode maps are small enough to label individually - which is most of their value. Matplotlib does not implement the Unicode bidirectional algorithm, so right-to-left text is drawn reversed. The cell below monkey-patches `matplotlib.text.Text.set_text` once so that any string containing Hebrew characters is converted to display order via `python-bidi` before it is drawn, and selects a font with Hebrew glyphs (Arial on Windows, DejaVu Sans elsewhere). It is idempotent - re-running does not stack patches. This is the same cell used in notebooks 02 and 03.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. Locating earlier stages

Stage folders are resolved by **two-digit prefix** (`OUT.glob('01*')`) rather than by exact slug, so a renamed folder does not break the notebook. `find_artifact` raises a `FileNotFoundError` that names the notebook to run, instead of failing later with an opaque pandas error.

Only one input is genuinely required - `stops_clean.csv` from notebook 01, which supplies the station names, coordinates and `region`/`metro` labels. The artifacts of notebooks 02 and 14 are loaded **if present** and used only for cross-checks; their absence downgrades two optional sections and nothing else.

In [ ]:
# --- Resolve earlier stages by two-digit prefix ---------------------------
def stage_dir(prefix):
    """Return the output folder whose name starts with `prefix` (e.g. '02'), or None."""
    matches = sorted(p for p in OUT.glob(prefix + '*') if p.is_dir())
    return matches[0] if matches else None


def find_artifact(prefix, filename, notebook_hint, required=True):
    """Locate `filename` inside the stage folder `prefix*`; explain how to produce it."""
    sd = stage_dir(prefix)
    if sd is not None:
        direct = sd / filename
        if direct.exists():
            return direct
        matches = sorted(sd.rglob(filename))
        if matches:
            return matches[0]
    if required:
        raise FileNotFoundError(
            f"{filename} was not found under {OUT / (prefix + '*')} - "
            f"run notebook {notebook_hint} first; it writes {filename}."
        )
    return None


stops_path = find_artifact('01', 'stops_clean.csv', '01_data_preparation')
stops_df = pd.read_csv(stops_path, dtype=str, keep_default_na=False, encoding='utf-8-sig')


def _to_float(x):
    """Coordinates arrive as strings and may be blank; return None instead of raising."""
    try:
        v = float(x)
    except (TypeError, ValueError):
        return None
    return v if np.isfinite(v) else None


ATTR = {
    r['stop_id']: {
        'stop_name': r.get('stop_name', '') or '',
        'lat': _to_float(r.get('stop_lat')),
        'lon': _to_float(r.get('stop_lon')),
        'region': r.get('region', '') or '',
        'metro': r.get('metro', '') or '',
        'parent_station': r.get('parent_station', '') or '',
    }
    for r in stops_df.to_dict('records')
}
DEFAULT_ATTR = {'stop_name': '', 'lat': None, 'lon': None,
                'region': '', 'metro': '', 'parent_station': ''}

# Optional cross-check inputs.
nodes_path = find_artifact('02', 'nodes.csv', '02_graph_construction', required=False)
edges_path = find_artifact('02', 'edges.csv', '02_graph_construction', required=False)
inventory14_path = find_artifact('14', 'mode_inventory.csv', '14_mode_inventory', required=False)

print(f'stop attributes : {len(ATTR):,} stops  <-  {stops_path}')
print('optional nodes.csv        :', nodes_path)
print('optional edges.csv        :', edges_path)
print('optional mode_inventory   :', inventory14_path)

## 5. Mode assignment: which trips belong to which mode

GTFS puts the mode on the **route**, not on the trip and certainly not on the stop time. To label a stop-time row with a mode we therefore need a two-step join, both steps of which fit comfortably in memory:

`stop_times.trip_id` -> `trips.route_id` -> `routes.route_type`

`routes.txt` is ~7.8k rows and `trips.txt` ~420k rows, so we build the dictionary `trip_mode: trip_id -> route_type` (420k entries, a few tens of MB) up front. The streaming pass then labels each of the 15.7M stop-time rows with a single dictionary lookup.

We also keep the agency name for each route, because it is what makes the minor modes legible: `route_type = 8` is nominally "trolleybus" in the GTFS specification, but the agencies behind those eight routes are taxi companies (*Metro Kav*, *Odelia Taxis*, *Rav-Kavit 4-5*), i.e. these are **shared-taxi lines coded under a spare route_type**, not trolleybuses. Reporting them as "trolleybus" without that caveat would be wrong, so the caveat is carried through every table in this notebook.

In [ ]:
# --- routes.txt / trips.txt / agency.txt ----------------------------------
for p in (DATA / 'routes.txt', DATA / 'trips.txt', DATA / 'agency.txt'):
    if not p.exists():
        raise FileNotFoundError(f'{p} is missing - the GTFS feed must be present under {DATA}.')

agency_name = {}
with open(DATA / 'agency.txt', encoding='utf-8-sig', newline='') as f:
    for a in csv.DictReader(f):
        agency_name[a.get('agency_id', '')] = a.get('agency_name', '')

route_type_of, route_meta = {}, {}
with open(DATA / 'routes.txt', encoding='utf-8-sig', newline='') as f:
    for r in csv.DictReader(f):
        try:
            rt = int(r['route_type'])
        except (TypeError, ValueError, KeyError):
            continue
        rid = r['route_id']
        route_type_of[rid] = rt
        route_meta[rid] = {
            'route_type': rt,
            'route_short_name': r.get('route_short_name', '') or '',
            'route_long_name': r.get('route_long_name', '') or '',
            'agency': agency_name.get(r.get('agency_id', ''), r.get('agency_id', '')),
        }

trip_mode, trip_route = {}, {}
trips_per_mode, trips_by_route = Counter(), Counter()
routes_with_trips = defaultdict(set)
with open(DATA / 'trips.txt', encoding='utf-8-sig', newline='') as f:
    for t in csv.DictReader(f):
        rt = route_type_of.get(t.get('route_id'))
        if rt is None:
            continue
        tid = t['trip_id']
        trip_mode[tid] = rt
        trips_per_mode[rt] += 1
        trips_by_route[t['route_id']] += 1
        routes_with_trips[rt].add(t['route_id'])
        if rt in MINOR_ROUTE_TYPES:
            trip_route[tid] = t['route_id']

routes_per_mode = Counter(route_type_of.values())
inventory = pd.DataFrame([
    {'route_type': rt,
     'mode_label': MODE_LABELS.get(rt, f'route_type {rt}'),
     'routes': routes_per_mode[rt],
     'routes_with_trips': len(routes_with_trips[rt]),
     'trips': trips_per_mode[rt],
     'trips_per_route': round(trips_per_mode[rt] / max(routes_per_mode[rt], 1), 1),
     'is_minor_mode': rt in MINOR_ROUTE_TYPES}
    for rt in sorted(routes_per_mode)
]).sort_values('trips', ascending=False).reset_index(drop=True)

print(f'routes: {len(route_type_of):,} | trips: {len(trip_mode):,}')
print(f'minor-mode trips to be streamed in detail: {sum(trips_per_mode[rt] for rt in MINOR_ROUTE_TYPES):,}')
inventory

### 5a. The minor-mode route list, and an optional cross-check against notebook 14

Thirty-four routes carry the entire minor-mode universe, so we can simply print all of them with their operator and trip count. This is the cheapest possible sanity check that the mode join is correct: the `route_type = 0` rows should be recognisable light-rail corridors, the `route_type = 5` rows the two Haifa cable systems, and so on. It also exposes the GTFS convention that each direction of a line is a **separate `route_id`** - which becomes structurally important two sections down.

If notebook 14 has been run, its `mode_inventory.csv` is compared against the counts computed here; a mismatch means the two notebooks disagree about the feed and should be investigated rather than silently averaged. If notebook 14 has not been run, the check is skipped.

In [ ]:
# --- Every minor-mode route, with operator and trip count -----------------
minor_routes = pd.DataFrame([
    {'route_type': meta['route_type'],
     'mode_label': MODE_LABELS.get(meta['route_type'], ''),
     'route_id': rid,
     'agency': meta['agency'],
     'route_short_name': meta['route_short_name'],
     'route_long_name': meta['route_long_name'],
     'trips': trips_by_route.get(rid, 0)}
    for rid, meta in route_meta.items() if meta['route_type'] in MINOR_ROUTE_TYPES
]).sort_values(['route_type', 'trips'], ascending=[True, False]).reset_index(drop=True)
minor_routes.to_csv(TABLES / 'minor_mode_routes.csv', index=False, encoding='utf-8-sig')

print('agencies per minor mode:')
for rt in MINOR_ROUTE_TYPES:
    ags = sorted(set(minor_routes.loc[minor_routes['route_type'] == rt, 'agency']))
    print(f'  {rt:>3} {MODE_LABELS[rt]:<18} {len(ags)} operator(s): {", ".join(ags)}')

# Optional consistency check against notebook 14.
if inventory14_path is not None:
    inv14 = pd.read_csv(inventory14_path, encoding='utf-8-sig')
    merged = inventory.merge(inv14, on='route_type', how='inner', suffixes=('_here', '_nb14'))
    if len(merged):
        bad = merged[(merged['routes_here'] != merged['routes_nb14'])
                     | (merged['trips_here'] != merged['trips_nb14'])]
        print('\nCross-check vs notebook 14 mode_inventory.csv:',
              'consistent' if bad.empty else 'MISMATCH - investigate')
        if not bad.empty:
            display(bad)
else:
    print('\nnotebook 14 output not present - inventory cross-check skipped (not an error).')

minor_routes

## 6. External data dependency: `stop_times.txt`

The stop sequences - and therefore the edges - live only in `stop_times.txt`, which is 816 MB and far above GitHub's file-size limit, so it is **not** in the repository. The cell below downloads it from Google Drive on first run and skips the download if the file is already present. This is the notebook's only external network dependency; everything else is either in the repo or produced by notebook 01. The download takes a few minutes on a first Colab run.

In [ ]:
# stop_times.txt is 816MB and is not tracked in git - fetch it on demand.
_ensure("gdown")
import gdown
STOP_TIMES = DATA / "stop_times.txt"
if not STOP_TIMES.exists():
    gdown.download(id="1V_yPAWXV6mGTFGrfiosah5LngcLZnviW",
                   output=str(STOP_TIMES), quiet=False)
print("stop_times.txt:", round(STOP_TIMES.stat().st_size / 1024**2, 1), "MB")

## 7. Parsing GTFS clock times - the >= 24:00 trap

GTFS times are **not** wall-clock times. They are offsets from the *service day's* noon-minus-twelve-hours, so a trip that runs past midnight is written `24:15:00`, `25:04:00` and so on rather than rolling over to `00:15:00`. In this feed about 1.1% of rows carry an hour of 24 or more, and the light rail alone contributes over a thousand of them.

The consequence is blunt: `datetime.strptime(value, '%H:%M:%S')` raises `ValueError: unconverted data remains` / `hour must be in 0..23` on those rows and would crash the pass. So we never construct a `datetime`. `gtfs_seconds` parses the three fields as integers and returns **seconds since service midnight**, which may exceed 86,400 - exactly the representation we want for measuring headways and service spans, because it keeps a 25:04 departure *after* a 23:50 departure instead of wrapping it to the start of the day.

The asserts below pin the behaviour, including the late-night case and the two malformed cases.

In [ ]:
# --- GTFS clock -> seconds since service midnight -------------------------
def gtfs_seconds(value):
    """Parse a GTFS 'HH:MM:SS' field into seconds since service midnight.

    Hours >= 24 are legal in GTFS ('25:30:00' is 01:30 on the next calendar day) and
    are preserved as > 86400 seconds. datetime/strptime must never be used here - it
    raises on hour >= 24, which is ~1.1% of the rows in this feed.
    """
    if not value:
        return None
    parts = value.strip().split(':')
    if len(parts) != 3:
        return None
    try:
        h, m, s = int(parts[0]), int(parts[1]), int(parts[2])
    except ValueError:
        return None
    return h * 3600 + m * 60 + s


def hhmm(sec):
    """Format seconds-since-service-midnight as HH:MM, keeping hours >= 24 visible."""
    if sec is None:
        return ''
    return f'{int(sec) // 3600:02d}:{int(sec) % 3600 // 60:02d}'


assert gtfs_seconds('05:10:00') == 5 * 3600 + 10 * 60
assert gtfs_seconds('25:30:00') == 25 * 3600 + 30 * 60 == 91800
assert gtfs_seconds('24:00:00') == 86400
assert gtfs_seconds('') is None and gtfs_seconds('not a time') is None
assert hhmm(91800) == '25:30'
print('gtfs_seconds OK -  "25:30:00" ->', gtfs_seconds('25:30:00'),
      'seconds since service midnight, formatted as', hhmm(gtfs_seconds('25:30:00')))

## 8. The streaming pass over `stop_times.txt`

This is the one expensive cell. The file is read with `csv.reader` one row at a time - it is never loaded into a DataFrame - and each row is labelled with its mode through the `trip_mode` dictionary built in section 5. Two different things are then recorded:

1. **For every mode, including bus and rail**: the number of stop calls and the set of distinct stops served. These two cheap counters are what let section 10 compare the minor modes against a bus baseline without a second pass over the file.
2. **For the four minor modes only**: the full detail of every stop call - `(stop_sequence, stop_id, departure_seconds)` grouped by `trip_id`. That is only about 88,600 rows out of 15.7M (the minor modes are 0.6% of the feed), so keeping them in memory costs a few MB and buys us exact stop sequences and exact departure times.

Note the difference from notebook 02, which had to build edges *incrementally* because it could not hold 15.7M rows. Here we can afford to buffer the minor-mode rows and sort each trip by `stop_sequence` afterwards, so this notebook does **not** rely on the feed being sorted by `(trip_id, stop_sequence)` - it only relies on `stop_sequence` being correct within a trip. That is a weaker assumption than notebook 02 makes.

Rows whose `trip_id` is absent from `trips.txt` are counted and skipped rather than silently dropped.

In [ ]:
# --- Single streaming pass over 15.7M rows --------------------------------
def stream_stop_times(path, trip_mode, minor_types, progress_every=PROGRESS_EVERY):
    """One pass over stop_times.txt.

    Returns:
      calls       : {trip_id: [(stop_sequence, stop_id, departure_seconds), ...]} - MINOR modes only
      mode_calls  : {route_type: number of stop-time rows}          - all modes
      mode_stops  : {route_type: set of stop_ids}                   - all modes
      stats       : counters describing the pass
    """
    minor = set(minor_types)
    calls = defaultdict(list)
    mode_calls = Counter()
    mode_stops = defaultdict(set)
    rows_read = unknown_trip_rows = bad_time_rows = bad_seq_rows = 0
    t0 = time.time()

    with open(path, encoding='utf-8-sig', newline='') as f:
        reader = csv.reader(f)
        header = next(reader)
        ti = header.index('trip_id')
        si = header.index('stop_id')
        qi = header.index('stop_sequence')
        di = header.index('departure_time')
        for row in reader:
            rows_read += 1
            trip = row[ti]
            mode = trip_mode.get(trip)
            if mode is None:
                unknown_trip_rows += 1
            else:
                stop = row[si]
                mode_calls[mode] += 1
                mode_stops[mode].add(stop)
                if mode in minor:
                    try:
                        seq = int(row[qi])
                    except (ValueError, IndexError):
                        seq = len(calls[trip])   # fall back to file order
                        bad_seq_rows += 1
                    dep = gtfs_seconds(row[di]) if di < len(row) else None
                    if dep is None:
                        bad_time_rows += 1
                    calls[trip].append((seq, stop, dep))
            if progress_every and rows_read % progress_every == 0:
                kept = sum(len(v) for v in calls.values())
                print(f'    {rows_read:,} rows | {kept:,} minor-mode calls kept '
                      f'| {time.time() - t0:,.0f}s')

    stats = {
        'stop_times_rows': rows_read,
        'rows_with_unknown_trip': unknown_trip_rows,
        'minor_mode_trips': len(calls),
        'minor_mode_calls': sum(len(v) for v in calls.values()),
        'unparsable_departure_times': bad_time_rows,
        'unparsable_stop_sequences': bad_seq_rows,
        'elapsed_seconds': round(time.time() - t0, 1),
    }
    return calls, mode_calls, mode_stops, stats


print(f'Streaming {STOP_TIMES.name} (15.7M rows) - this is the only slow cell ...')
calls, mode_calls, mode_stops, stream_stats = stream_stop_times(
    STOP_TIMES, trip_mode, MINOR_ROUTE_TYPES)

print('\nPass statistics:')
for k, v in stream_stats.items():
    print(f'  {k}: {v:,}' if isinstance(v, int) else f'  {k}: {v}')
print('\nStop calls and distinct stops per mode (whole feed):')
for rt in sorted(mode_calls, key=lambda r: -mode_calls[r]):
    print(f'  {rt:>3} {MODE_LABELS.get(rt, "?"):<18} calls={mode_calls[rt]:>12,}  stops={len(mode_stops[rt]):>7,}')

## 9. Building one graph per minor mode

The model is exactly the trip-adjacency graph defined in notebook 02, restricted to a single mode: a node is a stop served by that mode, and a directed edge `u -> v` exists when some trip of that mode calls at `v` immediately after `u`, weighted by the number of such trips. The undirected projection sums the two directions, so an undirected weight keeps its meaning of "services crossing this link in either direction".

Restricting to one mode is not the same as taking the induced subgraph of the national graph on that mode's stops: a bus segment between two light-rail stops would appear in the induced subgraph but is not a light-rail link. We build from the mode's own trips, which is the correct construction.

Alongside the graphs we keep, per mode and per stop, the number of stop calls, the set of routes serving it, and the list of departure times - the raw material for the intensity and headway sections.

In [ ]:
# --- Per-mode trip-adjacency graphs ---------------------------------------
def build_mode_structures(calls, trip_mode, trip_route, minor_types, attr):
    """Turn buffered stop calls into one directed + one undirected graph per mode."""
    edge_count = {rt: defaultdict(int) for rt in minor_types}
    stop_calls = {rt: Counter() for rt in minor_types}
    stop_routes = {rt: defaultdict(set) for rt in minor_types}
    stop_departures = {rt: defaultdict(list) for rt in minor_types}
    trips_seen = Counter()
    self_loops = Counter()
    one_stop_trips = Counter()

    for trip_id, rows in calls.items():
        rt = trip_mode[trip_id]
        route = trip_route.get(trip_id, '')
        trips_seen[rt] += 1
        ordered = sorted(rows, key=lambda r: r[0])
        if len(ordered) < 2:
            one_stop_trips[rt] += 1
        for _, stop, dep in ordered:
            stop_calls[rt][stop] += 1
            stop_routes[rt][stop].add(route)
            if dep is not None:
                stop_departures[rt][stop].append(dep)
        seq = [stop for _, stop, _ in ordered]
        for u, v in zip(seq, seq[1:]):
            if u == v:
                self_loops[rt] += 1        # same stop twice in a row: no connectivity info
            else:
                edge_count[rt][(u, v)] += 1

    graphs = {}
    for rt in minor_types:
        D = nx.DiGraph()
        for (u, v), c in edge_count[rt].items():
            D.add_edge(u, v, weight=c)
        G = nx.Graph()
        for u, v, data in D.edges(data=True):
            if G.has_edge(u, v):
                G[u][v]['weight'] += data['weight']
            else:
                G.add_edge(u, v, weight=data['weight'])
        for graph in (G, D):
            for n in graph.nodes():
                graph.nodes[n].update(attr.get(n, DEFAULT_ATTR))
        graphs[rt] = {'G': G, 'D': D}
    return graphs, stop_calls, stop_routes, stop_departures, trips_seen, self_loops, one_stop_trips


(graphs, stop_calls, stop_routes, stop_departures,
 trips_seen, self_loops, one_stop_trips) = build_mode_structures(
    calls, trip_mode, trip_route, MINOR_ROUTE_TYPES, ATTR)

for rt in MINOR_ROUTE_TYPES:
    G, D = graphs[rt]['G'], graphs[rt]['D']
    served = len(stop_calls[rt])
    print(f'{MODE_LABELS[rt]:<18} trips={trips_seen[rt]:>5,} | stops served={served:>4} | '
          f'graph nodes={G.number_of_nodes():>4} | directed edges={D.number_of_edges():>4} | '
          f'undirected edges={G.number_of_edges():>4} | self-loops skipped={self_loops[rt]}')
    if served != G.number_of_nodes():
        print(f'    note: {served - G.number_of_nodes()} stop(s) are served but have no '
              f'neighbour (single-stop trips: {one_stop_trips[rt]}) and are therefore not nodes')

## 10. Service intensity: the point of this notebook

This is where the minor modes stop looking minor. Two ratios are computed for **all six** modes, so that bus and rail act as a baseline:

* **trips per route** = scheduled trips / routes. How hard each line is worked.
* **calls per stop** = stop-time rows / distinct stops served. How much service the average station of that mode sees per service day.

Both are supply-side measures. GTFS carries no ridership, so nothing here says how many *people* use these services - only how much service is scheduled. That limitation is inherited by every criticality statement in this notebook and is stated again in the takeaways.

One honest wrinkle to watch for in the output: *calls per stop* for light rail is not dramatically above the bus average, and the reason is a data artefact rather than a fact about trams - the feed gives each direction of a light-rail line its own platform `stop_id`, so the service of one physical station is split across two rows. Section 11 measures that duplication explicitly; the per-**station** figure is roughly twice the per-`stop_id` figure. *Trips per route* is not affected by the artefact and is the cleaner comparison.

In [ ]:
# --- Service intensity for every mode -------------------------------------
intensity_rows = []
for rt in sorted(set(list(ALL_ROUTE_TYPES) + list(mode_calls))):
    stops_n = len(mode_stops.get(rt, ()))
    routes_n = routes_per_mode.get(rt, 0)
    trips_n = trips_per_mode.get(rt, 0)
    intensity_rows.append({
        'route_type': rt,
        'mode_label': MODE_LABELS.get(rt, f'route_type {rt}'),
        'routes': routes_n,
        'trips': trips_n,
        'stops': stops_n,
        'stop_calls': mode_calls.get(rt, 0),
        'trips_per_route': round(trips_n / routes_n, 1) if routes_n else np.nan,
        'calls_per_stop': round(mode_calls.get(rt, 0) / stops_n, 1) if stops_n else np.nan,
        'stops_per_route': round(stops_n / routes_n, 1) if routes_n else np.nan,
        'is_minor_mode': rt in MINOR_ROUTE_TYPES,
    })
intensity = pd.DataFrame(intensity_rows).sort_values('trips_per_route', ascending=False)
intensity.to_csv(TABLES / 'mode_service_intensity.csv', index=False, encoding='utf-8-sig')

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
panels = [('trips', 'Scheduled trips per mode (log)'),
          ('trips_per_route', 'Trips per route - service intensity'),
          ('calls_per_stop', 'Stop calls per stop_id')]
for ax, (col, title) in zip(axes, panels):
    d = intensity.sort_values(col, ascending=True)
    colors = [MODE_COLORS.get(rt, '#94a3b8') for rt in d['route_type']]
    bars = ax.barh(range(len(d)), d[col].to_numpy(), color=colors)
    ax.set_yticks(range(len(d)))
    ax.set_yticklabels(d['mode_label'], fontsize=9)
    ax.set_title(title, fontsize=11)
    if col == 'trips':
        ax.set_xscale('log')
    for bar, val in zip(bars, d[col].to_numpy()):
        ax.text(bar.get_width(), bar.get_y() + bar.get_height() / 2,
                f'  {val:,.0f}', va='center', fontsize=8)
    ax.margins(x=0.18)
fig.suptitle('Minor modes are few routes worked very hard (bus / rail shown for scale)', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES / 'service_intensity.png', dpi=FIG_DPI)
plt.show()

intensity

## 11. Platforms versus stations - a data artefact that changes the topology

Before measuring shape we have to be explicit about what a node *is* for these modes. In this feed a light-rail line is published as two `route_id`s (one per direction), and each direction has its **own** platform `stop_id`, with the two platforms of a physical station carrying the same `stop_name`. The cable tram, by contrast, uses a single `stop_id` per station for both directions.

The consequence is structural, not cosmetic: because the up-direction and the down-direction platforms never appear in the same trip, no edge ever joins them, and a single physical light-rail line appears in the graph as **two disjoint one-way corridors**. Any statement of the form "the light-rail network has N components" is therefore partly a statement about GTFS publishing conventions.

The cell below quantifies the duplication - distinct `stop_id` versus distinct `stop_name` per mode - so that every later count can be read with the right factor in mind. We deliberately do **not** merge platforms by name: name collisions between unrelated stops elsewhere in the country make name-merging unsafe, and the graph we analyse must stay the same object the rest of the project uses.

In [ ]:
# --- Platform duplication: stop_ids vs physical station names -------------
dup_rows = []
for rt in MINOR_ROUTE_TYPES:
    ids = set(stop_calls[rt])
    names = {ATTR.get(s, DEFAULT_ATTR)['stop_name'] for s in ids}
    with_parent = sum(1 for s in ids if ATTR.get(s, DEFAULT_ATTR)['parent_station'])
    dup_rows.append({
        'route_type': rt,
        'mode_label': MODE_LABELS[rt],
        'stop_ids': len(ids),
        'distinct_stop_names': len(names),
        'stop_ids_per_name': round(len(ids) / max(len(names), 1), 2),
        'stop_ids_with_parent_station': with_parent,
    })
platform_dup = pd.DataFrame(dup_rows)
platform_dup.to_csv(TABLES / 'platform_duplication.csv', index=False, encoding='utf-8-sig')

for r in platform_dup.itertuples():
    verdict = ('directional platforms are separate stop_ids'
               if r.stop_ids_per_name > 1.5 else 'one stop_id per physical station')
    print(f'{r.mode_label:<18} {r.stop_ids:>4} stop_ids / {r.distinct_stop_names:>4} names '
          f'= {r.stop_ids_per_name:.2f}  ->  {verdict}')
platform_dup

## 12. Size, shape and redundancy of each mode graph

Now the structural characterisation. For each mode we compute, exactly:

* **Size**: nodes, directed and undirected edges, number of connected components, largest-component share, average and maximum degree.
* **Redundancy**, via the **cyclomatic number** (circuit rank) `mu = m - n + c`, the number of independent cycles in a graph with `n` nodes, `m` undirected edges and `c` components. `mu = 0` means the graph is a forest: **between any two stations there is exactly one route, and every internal link is a bridge**. `mu > 0` means at least one alternative path exists somewhere. For transport networks this is the cleanest single redundancy statistic.
* **Shape class** per component, from the degree sequence:
  * `isolated` - a single node;
  * `path` - a tree with maximum degree <= 2 (a line);
  * `cycle` - every node of degree 2 (a loop);
  * `star` - a tree with one node adjacent to all others (a hub-and-spoke terminal);
  * `tree` - acyclic with branching but no loops;
  * `meshed` - contains at least one cycle.
* **Articulation points** and **bridges** - the exact single points of failure, using the same linear-time routines as notebook 03.
* **Exact betweenness** (Brandes, normalised, unweighted). Notebook 04 had to sample on the national graph; here `n` is at most a few hundred, so exact values are computed in milliseconds and there is no sampling error to caveat.
* **Service span and headway**: first and last departure in seconds since service midnight (so a 25:04 departure sorts after 23:50), and the median gap between consecutive departures at the busiest stop of the mode - a direct read of how frequent the service is.

Diameter is reported for the largest component only, and only when it is connected and has more than one node.

In [ ]:
# --- Structural characterisation of each minor mode -----------------------
def classify_component(H):
    """Name the shape of one connected component from its degree sequence."""
    n, m = H.number_of_nodes(), H.number_of_edges()
    if n == 1:
        return 'isolated'
    degs = sorted(d for _, d in H.degree())
    acyclic = (m == n - 1)
    if acyclic:
        if degs[-1] <= 2:
            return 'path'
        if degs[-1] == n - 1:
            return 'star'
        return 'tree'
    if all(d == 2 for d in degs):
        return 'cycle'
    return 'meshed'


def headway_minutes(departure_lists):
    """Median gap between consecutive departures at the busiest stop, in minutes."""
    if not departure_lists:
        return np.nan
    busiest = max(departure_lists.values(), key=len)
    if len(busiest) < 2:
        return np.nan
    d = np.diff(np.sort(np.asarray(busiest, dtype=float)))
    d = d[d > 0]
    return round(float(np.median(d)) / 60.0, 1) if len(d) else np.nan


BETW = {}
summary_rows, component_rows = [], []
for rt in MINOR_ROUTE_TYPES:
    G, D = graphs[rt]['G'], graphs[rt]['D']
    n, m = G.number_of_nodes(), G.number_of_edges()
    comps = sorted(nx.connected_components(G), key=len, reverse=True)
    degs = np.array([d for _, d in G.degree()]) if n else np.array([0])
    ap = sorted(set(nx.articulation_points(G))) if n else []
    br = list(nx.bridges(G)) if n else []
    mu = m - n + len(comps) if n else 0
    BETW[rt] = nx.betweenness_centrality(G, normalized=True) if (n and EXACT_BETWEENNESS) else {}

    shapes = []
    for i, c in enumerate(comps):
        H = G.subgraph(c)
        shape = classify_component(H)
        shapes.append(shape)
        try:
            diam = nx.diameter(H) if len(c) > 1 else 0
        except (nx.NetworkXError, nx.NetworkXNoPath):
            diam = np.nan
        sample = sorted((G.nodes[x]['stop_name'] or x) for x in c)[:3]
        component_rows.append({
            'route_type': rt, 'mode_label': MODE_LABELS[rt], 'component_id': i,
            'nodes': len(c), 'edges': H.number_of_edges(), 'shape': shape,
            'cyclomatic_number': H.number_of_edges() - len(c) + 1,
            'diameter': diam,
            'metro': Counter(G.nodes[x]['metro'] for x in c).most_common(1)[0][0],
            'region': Counter(G.nodes[x]['region'] for x in c).most_common(1)[0][0],
            'sample_stations': ' | '.join(sample),
        })

    deps_all = [d for lst in stop_departures[rt].values() for d in lst]
    calls_total = mode_calls.get(rt, 0)
    stops_total = len(stop_calls[rt])
    summary_rows.append({
        'route_type': rt,
        'mode_label': MODE_LABELS[rt],
        'routes': routes_per_mode.get(rt, 0),
        'trips': trips_per_mode.get(rt, 0),
        'stops': stops_total,
        'graph_nodes': n,
        'directed_edges': D.number_of_edges(),
        'undirected_edges': m,
        'components': len(comps),
        'largest_component_nodes': len(comps[0]) if comps else 0,
        'largest_component_share': round(len(comps[0]) / n, 4) if n else np.nan,
        'avg_degree': round(float(degs.mean()), 2),
        'max_degree': int(degs.max()),
        'density': round(nx.density(G), 5) if n > 1 else np.nan,
        'cyclomatic_number': mu,
        'edges_per_node': round(m / n, 3) if n else np.nan,
        'articulation_points': len(ap),
        'articulation_point_share': round(len(ap) / n, 4) if n else np.nan,
        'bridges': len(br),
        'bridge_share': round(len(br) / m, 4) if m else np.nan,
        'shape_class': ', '.join(f'{v}x {k}' for k, v in Counter(shapes).most_common()),
        'diameter_largest_component': component_rows[-len(comps)]['diameter'] if comps else np.nan,
        'trips_per_route': round(trips_per_mode.get(rt, 0) / routes_per_mode.get(rt, 1), 1),
        'stop_calls': calls_total,
        'calls_per_stop': round(calls_total / stops_total, 1) if stops_total else np.nan,
        'max_calls_at_a_stop': max(stop_calls[rt].values()) if stops_total else 0,
        'first_departure': hhmm(min(deps_all)) if deps_all else '',
        'last_departure': hhmm(max(deps_all)) if deps_all else '',
        'service_span_hours': round((max(deps_all) - min(deps_all)) / 3600, 1) if deps_all else np.nan,
        'departures_after_midnight': int(sum(1 for d in deps_all if d >= 86400)),
        'median_headway_busiest_stop_minutes': headway_minutes(stop_departures[rt]),
        'max_exact_betweenness': round(max(BETW[rt].values()), 4) if BETW[rt] else np.nan,
        'exact_betweenness': bool(EXACT_BETWEENNESS),
    })

minor_summary = pd.DataFrame(summary_rows)
minor_summary.to_csv(TABLES / 'minor_modes_summary.csv', index=False, encoding='utf-8-sig')
components_df = pd.DataFrame(component_rows)
components_df.to_csv(TABLES / 'minor_mode_components.csv', index=False, encoding='utf-8-sig')

print('saved:', TABLES / 'minor_modes_summary.csv')
print('saved:', TABLES / 'minor_mode_components.csv')
display(minor_summary[['mode_label', 'graph_nodes', 'undirected_edges', 'components',
                       'avg_degree', 'max_degree', 'cyclomatic_number', 'shape_class',
                       'articulation_points', 'bridges']])
components_df

### 12a. Reading the redundancy numbers

The cell below turns the two structural columns into the sentence that matters for criticality. A mode whose cyclomatic number is zero is a **forest**: every edge is a bridge, every non-leaf node is an articulation point, and there is exactly one path between any two stations it can serve. On such a graph the question "which station is critical?" has a degenerate answer - *nearly all of them are* - and the only meaningful ranking is by **how much** traffic each cut would strand, which is what section 16 measures.

This is the structural difference from the bus network, where notebook 03 found articulation points at only about 3% of stations.

In [ ]:
# --- Redundancy verdict per mode ------------------------------------------
for r in minor_summary.itertuples():
    if r.graph_nodes == 0:
        print(f'{r.mode_label:<18} no graph (no multi-stop trips)')
        continue
    if r.cyclomatic_number == 0:
        verdict = ('FOREST - zero independent cycles: exactly one path between any two '
                   'stations, every link is a bridge')
    else:
        verdict = (f'{r.cyclomatic_number} independent cycle(s): some alternative paths exist')
    print(f'{r.mode_label:<18} n={r.graph_nodes:<4} m={r.undirected_edges:<4} '
          f'mu={r.cyclomatic_number:<3} AP={r.articulation_points} '
          f'({r.articulation_point_share:.0%} of stations)  bridges={r.bridges} '
          f'({r.bridge_share:.0%} of links)')
    print(f'{"":<18} -> {verdict}')

print()
print('For contrast, the national (mostly bus) graph in notebook 03 has articulation points')
print('at roughly 3% of its stations and bridges on under 2% of its links.')

## 13. Station-level metrics, with exact betweenness

This cell writes the contract table `tables/lightrail_station_metrics.csv`. It holds **one row per station of every minor mode**, not only light rail - the `mode_label` column is what separates them, and a consumer that wants light rail alone should filter on `mode_label == 'tram/light rail'`. The required columns come first and are named exactly as the data contract specifies:

`stop_id, stop_name, lat, lon, degree, weighted_degree, mode_label`

`weighted_degree` is the sum of the undirected edge weights at the station, i.e. **the number of scheduled services crossing that station per service day, counting both directions** - the natural load measure for this mode.

Additional columns are appended for the analysis here and are safe to ignore: exact betweenness, stop calls, the number of routes serving the station, the articulation-point flag, the component id, and `region`/`metro`. Stations are sorted by mode and then by descending exact betweenness.

In [ ]:
# --- Station metrics for every minor mode ---------------------------------
station_rows = []
for rt in MINOR_ROUTE_TYPES:
    G = graphs[rt]['G']
    if G.number_of_nodes() == 0:
        continue
    comp_of = {node: i for i, c in enumerate(
        sorted(nx.connected_components(G), key=len, reverse=True)) for node in c}
    ap = set(nx.articulation_points(G))
    wdeg = dict(G.degree(weight='weight'))
    for node, data in G.nodes(data=True):
        station_rows.append({
            'stop_id': node,
            'stop_name': data.get('stop_name', ''),
            'lat': data.get('lat'),
            'lon': data.get('lon'),
            'degree': G.degree(node),
            'weighted_degree': int(wdeg.get(node, 0)),
            'mode_label': MODE_LABELS[rt],
            'route_type': rt,
            'exact_betweenness': round(BETW[rt].get(node, 0.0), 6),
            'stop_calls': int(stop_calls[rt].get(node, 0)),
            'n_routes': len(stop_routes[rt].get(node, ())),
            'is_articulation_point': node in ap,
            'component_id': comp_of.get(node, -1),
            'region': data.get('region', ''),
            'metro': data.get('metro', ''),
        })

station_metrics = (pd.DataFrame(station_rows)
                   .sort_values(['route_type', 'exact_betweenness', 'weighted_degree'],
                                ascending=[True, False, False])
                   .reset_index(drop=True))
station_metrics.to_csv(TABLES / 'lightrail_station_metrics.csv', index=False, encoding='utf-8-sig')
print('saved:', TABLES / 'lightrail_station_metrics.csv',
      f'({len(station_metrics):,} stations across {station_metrics["mode_label"].nunique()} modes)')

missing_coords = station_metrics[['lat', 'lon']].isna().any(axis=1).sum()
print('stations with missing coordinates:', int(missing_coords))

print('\nTop light-rail stations by exact betweenness:')
display(station_metrics[station_metrics['route_type'] == 0]
        .head(TOP_N)[['stop_name', 'metro', 'degree', 'weighted_degree',
                      'exact_betweenness', 'stop_calls', 'is_articulation_point']])
print('Busiest stations of every minor mode (by services crossing the station):')
station_metrics.sort_values('weighted_degree', ascending=False).head(TOP_N)[
    ['mode_label', 'stop_name', 'metro', 'degree', 'weighted_degree', 'stop_calls']]

## 14. Geography of each mode

Four panels, one per mode, each drawing the actual links as line segments between station coordinates and the stations as points sized by the number of services crossing them. This is a plain longitude/latitude scatter rather than a projected map, so the aspect ratio of each panel is set to `1 / cos(mean latitude)` to keep distances approximately honest.

The panels answer the geographic half of the question: where do these modes actually exist? The short answer visible in the plots is that each one is a *single metropolitan artefact* - light rail in Jerusalem and Tel Aviv, cable tram in Haifa, shared taxis inside Tel Aviv, and the demand-responsive services in rural clusters (the Judean foothills and the Eilot region in the far south). None of them is a national network, which is precisely why the national aggregates cannot see them.

Stations without usable coordinates are excluded and counted; the test is for a present, finite value, so a coordinate of exactly `0.0` would be kept while `NaN` is dropped.

In [ ]:
# --- One geographic panel per mode ----------------------------------------
def has_coords(data):
    """True only when both coordinates are present and finite (0.0 included)."""
    lat, lon = data.get('lat'), data.get('lon')
    return (lat is not None and lon is not None
            and np.isfinite(lat) and np.isfinite(lon))


def draw_mode(ax, rt, label_stations=False, max_labels=MAX_LABELLED_STATIONS, nodes=None):
    """Draw one mode's graph on `ax`; returns the number of stations plotted."""
    G = graphs[rt]['G']
    sub = G.subgraph(nodes) if nodes is not None else G
    pts = {n: (d['lon'], d['lat']) for n, d in sub.nodes(data=True) if has_coords(d)}
    if not pts:
        ax.set_axis_off()
        ax.set_title(f'{MODE_LABELS[rt]} - no usable coordinates')
        return 0
    for u, v, d in sub.edges(data=True):
        if u in pts and v in pts:
            ax.plot([pts[u][0], pts[v][0]], [pts[u][1], pts[v][1]],
                    color=MODE_COLORS.get(rt, '#475569'), linewidth=1.6, alpha=0.75, zorder=1)
    load = np.array([stop_calls[rt].get(n, 0) for n in pts], dtype=float)
    sizes = 18 + 90 * (load / load.max()) if load.max() > 0 else np.full(len(pts), 25.0)
    xs = [p[0] for p in pts.values()]
    ys = [p[1] for p in pts.values()]
    ax.scatter(xs, ys, s=sizes, c=MODE_COLORS.get(rt, '#475569'),
               edgecolors='white', linewidths=0.6, zorder=2)
    if label_stations and len(pts) <= max_labels:
        placed = set()
        for n, (x, y) in pts.items():
            name = sub.nodes[n].get('stop_name', '') or n
            if name in placed:          # both directional platforms carry the same name
                continue
            placed.add(name)
            ax.annotate(name, (x, y), fontsize=7, xytext=(3, 3),
                        textcoords='offset points', zorder=3)
    ax.set_aspect(1 / np.cos(np.deg2rad(float(np.mean(ys)))))
    ax.set_xlabel('Longitude', fontsize=9)
    ax.set_ylabel('Latitude', fontsize=9)
    return len(pts)


fig, axes = plt.subplots(2, 2, figsize=(13, 13))
for ax, rt in zip(axes.ravel(), MINOR_ROUTE_TYPES):
    plotted = draw_mode(ax, rt)
    row = minor_summary.loc[minor_summary['route_type'] == rt].iloc[0]
    ax.set_title(f'{MODE_LABELS[rt]} (route_type {rt})\n'
                 f'{row.graph_nodes} stations, {row.undirected_edges} links, '
                 f'{row.components} component(s), {plotted} plotted', fontsize=10)
fig.suptitle('Each minor mode is a single metropolitan or rural artefact, not a national network',
             fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES / 'mode_network_maps.png', dpi=FIG_DPI)
plt.show()

### 14a. The light-rail lines, with station names

Light rail is small enough to label completely, which makes the corridor structure legible: each line is a chain of stations from one terminus to the other, with at most a short branch. The two panels split the light-rail graph by metropolitan area (`metro`), because the Jerusalem and Tel Aviv systems are entirely separate networks that merely share a `route_type`. Duplicate directional platforms are labelled once per name, per section 11.

The same treatment is given to the cable tram, whose two components - the Carmelit funicular and the Haifa cable car - are among the shortest "networks" it is possible to draw.

In [ ]:
# --- Labelled maps: light rail by metro, then cable tram ------------------
G0 = graphs[0]['G']
groups = defaultdict(list)
for n, d in G0.nodes(data=True):
    groups[d.get('metro') or d.get('region') or 'unknown'].append(n)
groups = dict(sorted(groups.items(), key=lambda kv: -len(kv[1])))

if groups:
    fig, axes = plt.subplots(1, len(groups), figsize=(7.5 * len(groups), 9.5))
    axes = np.atleast_1d(axes)
    for ax, (metro, nodes) in zip(axes, groups.items()):
        plotted = draw_mode(ax, 0, label_stations=True, nodes=nodes)
        sub = G0.subgraph(nodes)
        ncomp = nx.number_connected_components(sub)
        names = len({sub.nodes[n]['stop_name'] for n in nodes})
        ax.set_title(f'Light rail - {metro}\n{len(nodes)} platform stop_ids / ~{names} named '
                     f'stations, {ncomp} graph component(s)', fontsize=11)
    fig.suptitle('Light rail: each line appears twice, once per direction of travel', fontsize=13)
    plt.tight_layout()
    plt.savefig(FIGURES / 'lightrail_labeled_map.png', dpi=FIG_DPI)
    plt.show()

fig, ax = plt.subplots(figsize=(7.5, 8))
draw_mode(ax, 5, label_stations=True)
row5 = minor_summary.loc[minor_summary['route_type'] == 5].iloc[0]
ax.set_title(f'Cable tram (route_type 5) - {row5.graph_nodes} stations in '
             f'{row5.components} separate systems, {row5.trips:,} trips/day', fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES / 'cabletram_labeled_map.png', dpi=FIG_DPI)
plt.show()

## 15. Exact betweenness along each mode

Betweenness centrality counts the shortest paths that pass through a station. On a path graph it has a characteristic parabolic profile - the middle station lies on the most origin-destination pairs, the termini on none - and any departure from that parabola marks a junction or a branch.

These values are **exact** (Brandes, normalised, unweighted), not the `k`-sample approximation notebook 04 was forced to use on the 30k-node national graph, and there is no sampling caveat attached to them. The plot shows the sorted betweenness profile per mode and the leading stations are tabulated. The absolute magnitudes are not comparable to the national numbers, because betweenness is normalised within each graph: a station can be the most central object in the Carmelit and still be irrelevant nationally.

In [ ]:
# --- Exact betweenness profiles -------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

for rt in MINOR_ROUTE_TYPES:
    vals = sorted(BETW[rt].values(), reverse=True)
    if not vals:
        continue
    axes[0].plot(range(1, len(vals) + 1), vals, marker='o', markersize=3,
                 color=MODE_COLORS.get(rt), label=f'{MODE_LABELS[rt]} (n={len(vals)})')
axes[0].set_xlabel('Station rank within its mode')
axes[0].set_ylabel('Exact betweenness (normalised)')
axes[0].set_title('Exact betweenness profile - no sampling error')
axes[0].legend(fontsize=8)

top_bt = (station_metrics.sort_values('exact_betweenness', ascending=False)
          .head(TOP_N).iloc[::-1])
axes[1].barh(range(len(top_bt)),
             top_bt['exact_betweenness'].to_numpy(),
             color=[MODE_COLORS.get(rt, '#475569') for rt in top_bt['route_type']])
axes[1].set_yticks(range(len(top_bt)))
axes[1].set_yticklabels([f'{r.stop_name} [{r.mode_label}]' for r in top_bt.itertuples()],
                        fontsize=8)
axes[1].set_xlabel('Exact betweenness (normalised within mode)')
axes[1].set_title(f'Top {TOP_N} minor-mode stations by exact betweenness')
plt.tight_layout()
plt.savefig(FIGURES / 'exact_betweenness.png', dpi=FIG_DPI)
plt.show()

station_metrics.sort_values('exact_betweenness', ascending=False).head(TOP_N)[
    ['mode_label', 'stop_name', 'metro', 'degree', 'weighted_degree',
     'exact_betweenness', 'is_articulation_point']]

## 16. Exact single-station damage

On graphs this small we do not need a sampled attack simulation: we can remove **every** station in turn and measure the exact consequence. Two measures are recorded per station:

* `largest_component_share_surviving` - the size of the largest surviving component divided by `n - 1`. For a path graph, removing a station near the middle halves this immediately.
* `pairs_disconnected_share` - the fraction of station pairs that could reach each other before the removal (excluding pairs involving the removed station itself) and can no longer do so. This is the cleaner damage measure because it is insensitive to which side of the cut is larger. The baseline correctly excludes the removed node: if it sat in a component of size `k`, then `k - 1` pairs vanish by definition and are not counted as damage.

The cost is `O(n)` connectivity scans per mode - a few hundred operations in total. Note what this measures and what it does not: it is *topological* isolation within a single mode. It says nothing about a passenger's ability to switch to a bus, which is a multimodal question that notebook 17 addresses - and, as section 17 shows, one that this feed answers pessimistically, since light-rail stops share no `stop_id` with any bus stop.

In [ ]:
# --- Remove every station in turn (exact, no sampling) --------------------
def single_station_damage(G):
    """Exact damage caused by removing each node once."""
    comps = list(nx.connected_components(G))
    size_of = {n: len(c) for c in comps for n in c}
    base_pairs = sum(len(c) * (len(c) - 1) // 2 for c in comps)
    n = G.number_of_nodes()
    out = []
    for node in G.nodes():
        H = G.copy()
        H.remove_node(node)
        after = list(nx.connected_components(H))
        pairs_after = sum(len(c) * (len(c) - 1) // 2 for c in after)
        # pairs that existed among the OTHER nodes before the removal
        baseline = base_pairs - (size_of[node] - 1)
        largest = max((len(c) for c in after), default=0)
        out.append({
            'stop_id': node,
            'largest_component_share_surviving': round(largest / (n - 1), 4) if n > 1 else np.nan,
            'pairs_disconnected_share': round(1 - pairs_after / baseline, 4) if baseline else 0.0,
            'components_after': len(after),
        })
    return pd.DataFrame(out)


damage_frames = []
for rt in MINOR_ROUTE_TYPES:
    G = graphs[rt]['G']
    if G.number_of_nodes() < 2:
        continue
    d = single_station_damage(G)
    d['route_type'] = rt
    d['mode_label'] = MODE_LABELS[rt]
    damage_frames.append(d)

damage = pd.concat(damage_frames, ignore_index=True)
damage = damage.merge(
    station_metrics[['stop_id', 'route_type', 'stop_name', 'metro', 'degree',
                     'weighted_degree', 'exact_betweenness', 'is_articulation_point']],
    on=['stop_id', 'route_type'], how='left')
damage = damage.sort_values(['route_type', 'pairs_disconnected_share'],
                            ascending=[True, False]).reset_index(drop=True)
damage.to_csv(TABLES / 'minor_mode_single_station_damage.csv', index=False, encoding='utf-8-sig')

fig, ax = plt.subplots(figsize=(11, 8))
top_dmg = damage.sort_values('pairs_disconnected_share', ascending=False).head(TOP_N).iloc[::-1]
ax.barh(range(len(top_dmg)), top_dmg['pairs_disconnected_share'].to_numpy(),
        color=[MODE_COLORS.get(rt, '#475569') for rt in top_dmg['route_type']])
ax.set_yticks(range(len(top_dmg)))
ax.set_yticklabels([f'{r.stop_name} [{r.mode_label}]' for r in top_dmg.itertuples()], fontsize=8)
ax.set_xlabel('Share of station pairs disconnected by removing this single station')
ax.set_title(f'Exact single-station damage - top {TOP_N} minor-mode stations')
plt.tight_layout()
plt.savefig(FIGURES / 'single_station_damage.png', dpi=FIG_DPI)
plt.show()

print('Mean share of pairs disconnected by removing one station, per mode:')
print(damage.groupby('mode_label')['pairs_disconnected_share']
      .agg(['mean', 'max', 'count']).round(3).to_string())

## 17. Where do the minor modes sit inside the national graph?

An optional but revealing cross-check, run only if notebook 02 has produced `nodes.csv` and `edges.csv`. Notebook 03 reported that the national graph is not quite connected: about a dozen components, one of which holds ~99.3% of stations. This cell asks **what the other components are** by labelling every national component with the modes serving its stops.

The expected - and, in this feed, actual - answer is that the small components are precisely the minor modes plus rail. Light rail, cable tram and shared-taxi stops share **no** `stop_id` with any bus stop, so in a graph whose nodes are `stop_id`s they cannot touch the bus network at all: a passenger changing from tram to bus walks between two stops that the model treats as unrelated. The demand-responsive services are the exception - most of their stops are ordinary bus stops, so they are absorbed into the giant component.

Two consequences worth stating plainly. First, the "fragmentation" of the national network reported in notebook 03 is mostly an *encoding* effect of separate stop identifiers, not evidence of a disconnected transport system. Second, this notebook's damage numbers are a lower bound on real-world resilience and an upper bound on modelled connectivity: within-mode they are exact, but no substitution between modes is represented anywhere in the model.

In [ ]:
# --- Optional: locate the minor modes in the national graph ---------------
if nodes_path is None or edges_path is None:
    print('notebook 02 artifacts not found - national placement cross-check skipped '
          '(run 02_graph_construction to enable it).')
else:
    nat_edges = pd.read_csv(edges_path, dtype={'from_stop': str, 'to_stop': str},
                            encoding='utf-8-sig')
    NAT = nx.from_pandas_edgelist(nat_edges, 'from_stop', 'to_stop')
    nat_comps = sorted(nx.connected_components(NAT), key=len, reverse=True)

    stop_to_modes = defaultdict(set)
    for rt, stops in mode_stops.items():
        for s in stops:
            stop_to_modes[s].add(rt)

    rows = []
    for i, c in enumerate(nat_comps):
        counts = Counter()
        for s in c:
            for rt in stop_to_modes.get(s, ()):
                counts[rt] += 1
        rows.append({
            'component_rank': i,
            'nodes': len(c),
            'modes_present': ', '.join(f'{MODE_LABELS.get(rt, rt)}:{v}'
                                       for rt, v in counts.most_common()),
            'dominant_mode': MODE_LABELS.get(counts.most_common(1)[0][0], '?') if counts else '?',
            'sample_station': sorted(ATTR.get(s, DEFAULT_ATTR)['stop_name'] for s in c)[0],
        })
    nat_comp_df = pd.DataFrame(rows)
    nat_comp_df.to_csv(TABLES / 'national_component_modes.csv', index=False, encoding='utf-8-sig')

    print(f'national graph: {NAT.number_of_nodes():,} nodes, {len(nat_comps)} components')
    display(nat_comp_df)

    # How many stops of each minor mode are shared with another mode at all?
    print('Stops shared with at least one other mode (same stop_id):')
    for rt in MINOR_ROUTE_TYPES:
        stops = mode_stops.get(rt, set())
        shared = sum(1 for s in stops if len(stop_to_modes[s]) > 1)
        print(f'  {MODE_LABELS[rt]:<18} {shared:>4} / {len(stops):<4} '
              f'({shared / max(len(stops), 1):.0%}) - '
              f'{"integrated with the bus network" if shared else "no shared stop_id with any other mode"}')

## 18. When do these modes actually run?

The last dimension of intensity is time. Using the seconds-since-service-midnight values parsed in section 7, we count departures per service hour for each mode. Hours 24, 25, ... are kept as separate buckets rather than folded back to 0, 1, ... - they are late-night services of the *previous* service day, and merging them would both distort the small hours and hide the fact that these modes run past midnight at all.

The profile separates genuine mass-transit modes from the rest: a mode with a smooth all-day profile, a long span and a headway of a few minutes is carrying a metropolitan corridor; a mode with a couple of dozen departures clustered in office hours is a niche service whose "network" statistics should not be over-interpreted.

In [ ]:
# --- Departures per service hour ------------------------------------------
hour_rows = []
for rt in MINOR_ROUTE_TYPES:
    deps = [d for lst in stop_departures[rt].values() for d in lst]
    counts = Counter(d // 3600 for d in deps)
    for h in range(0, 28):
        hour_rows.append({'route_type': rt, 'mode_label': MODE_LABELS[rt],
                          'service_hour': h, 'departures': counts.get(h, 0)})
hourly = pd.DataFrame(hour_rows)
hourly.to_csv(TABLES / 'hourly_departures.csv', index=False, encoding='utf-8-sig')

fig, ax = plt.subplots(figsize=(12, 5.5))
for rt in MINOR_ROUTE_TYPES:
    d = hourly[hourly['route_type'] == rt]
    total = d['departures'].sum()
    if total == 0:
        continue
    ax.plot(d['service_hour'], d['departures'] / total, marker='o', markersize=3,
            color=MODE_COLORS.get(rt), label=f'{MODE_LABELS[rt]} ({total:,} calls)')
ax.axvline(24, color='#111827', linestyle='--', linewidth=1)
ax.text(24.1, ax.get_ylim()[1] * 0.9, 'GTFS hours >= 24\n(after midnight)', fontsize=8)
ax.set_xlabel('Service hour (seconds since service midnight // 3600)')
ax.set_ylabel('Share of the mode\'s stop calls')
ax.set_title('When each minor mode runs')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES / 'hourly_departures.png', dpi=FIG_DPI)
plt.show()

minor_summary[['mode_label', 'first_departure', 'last_departure', 'service_span_hours',
               'departures_after_midnight', 'median_headway_busiest_stop_minutes',
               'max_calls_at_a_stop']]

## 19. Saving the stage summary

The headline numbers are collected into `lightrail_summary.json` (the contract artifact for this stage), the four graphs are pickled as `{mode_label: networkx.Graph}` so a later notebook can reuse them without re-streaming the 816 MB feed, and every file written by this notebook is listed with its size as a final check that nothing landed outside the stage folder.

In [ ]:
# --- Stage summary and artifacts ------------------------------------------
lr = minor_summary.loc[minor_summary['route_type'] == 0].iloc[0]
bus_row = intensity.loc[intensity['route_type'] == 3]
bus_tpr = float(bus_row['trips_per_route'].iloc[0]) if len(bus_row) else None

summary = {
    'modes_analysed': {str(rt): MODE_LABELS[rt] for rt in MINOR_ROUTE_TYPES},
    'exact_centrality': bool(EXACT_BETWEENNESS),
    'stream_stats': stream_stats,
    'lightrail': {
        'routes': int(lr.routes), 'trips': int(lr.trips),
        'stop_ids': int(lr.graph_nodes),
        'distinct_station_names': int(platform_dup.loc[
            platform_dup['route_type'] == 0, 'distinct_stop_names'].iloc[0]),
        'undirected_edges': int(lr.undirected_edges),
        'components': int(lr.components),
        'cyclomatic_number': int(lr.cyclomatic_number),
        'articulation_points': int(lr.articulation_points),
        'articulation_point_share': float(lr.articulation_point_share),
        'bridges': int(lr.bridges),
        'trips_per_route': float(lr.trips_per_route),
        'max_calls_at_a_stop': int(lr.max_calls_at_a_stop),
        'median_headway_busiest_stop_minutes': (
            None if pd.isna(lr.median_headway_busiest_stop_minutes)
            else float(lr.median_headway_busiest_stop_minutes)),
        'service_span_hours': float(lr.service_span_hours),
        'shape_class': lr.shape_class,
    },
    'bus_trips_per_route_baseline': bus_tpr,
    'per_mode': {
        MODE_LABELS[rt]: {
            k: (None if pd.isna(v) else (v.item() if hasattr(v, 'item') else v))
            for k, v in minor_summary.loc[
                minor_summary['route_type'] == rt].iloc[0].to_dict().items()
        }
        for rt in MINOR_ROUTE_TYPES
    },
    'caveats': [
        'route_type 8 is labelled trolleybus in GTFS but is operated by shared-taxi companies.',
        'Light-rail directional platforms are separate stop_ids, so each line appears as two '
        'one-way components; component counts are partly a publishing convention.',
        'Weights are scheduled trips, not passengers - GTFS carries no ridership data.',
        'Damage measures are within-mode only; no modal substitution is represented.',
    ],
}
with open(STAGE / 'lightrail_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2, default=str)

with open(STAGE / 'lightrail_graphs.pkl', 'wb') as f:
    pickle.dump({MODE_LABELS[rt]: graphs[rt]['G'] for rt in MINOR_ROUTE_TYPES}, f)

print('Artifacts written under', STAGE)
for p in sorted(STAGE.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(STAGE)}  ({p.stat().st_size / 1024:,.1f} KB)')

## Takeaways

*(Numbers below are those printed by the cells above on this feed snapshot; re-run to refresh them.)*

* **The minor modes are 0.8% of the routes and 1.5% of the trips, but they are not small in intensity.** Cable tram runs ~752 trips per route and light rail ~361, against ~61 for bus and ~1.2 for rail. Measured per line, these are the hardest-worked services in the country: a single light-rail platform in Tel Aviv sees over 1,200 scheduled calls in a service day, i.e. a tram roughly every 70 seconds across the operating span, and the Haifa cable car stations see ~2,400 calls each.
* **They are corridors, not networks.** The light-rail graph is a **forest** - cyclomatic number 0 - so *every* link is a bridge and ~93% of its stations are articulation points; the cable tram is likewise two disjoint paths. Compare notebook 03's national figures: ~3% of stations are articulation points and under 2% of links are bridges. This is the structural point of the notebook: on a path there is no alternative route by construction, so the redundancy that makes the bus network absorb failures simply does not exist here.
* **Criticality therefore has to be measured by *how much* is stranded, not by *whether* something is stranded.** The exact single-station removals in section 16 show damage rising smoothly toward the middle of each corridor - removing a mid-line station disconnects roughly half of all station pairs on that line - while terminal stations cost almost nothing. Combined with the load figures, the mid-corridor light-rail stations are the highest-consequence single points of failure in the whole minor-mode set: high traffic, zero redundancy.
* **The one genuinely meshed minor mode is the demand-responsive service (route_type 715).** It has 153 stops, ~194 links and a cyclomatic number of ~43 - a real mesh with alternative paths - because different trip variants of the same rural route take different roads. That is an artefact of how flexible routing is published, not evidence of a redundant physical system, and it should not be read as "rural service is more resilient".
* **Two labelling problems are load-bearing and must not be ignored.** (1) `route_type = 8` is "trolleybus" in the GTFS specification but is operated here by taxi companies - 8 routes, 47 trips, and every trip has exactly two stop calls, so its "graph" is three disconnected stubs with no interior at all. No network conclusion should be drawn from it. (2) Each light-rail line is published as two directional `route_id`s with **separate platform `stop_id`s**, which is why 138 light-rail stop_ids correspond to only ~69 named stations and why the mode shows 4 components rather than 2 lines. Read every per-`stop_id` figure for light rail with that factor of two in mind.
* **The minor modes are exactly the fragmentation the national analysis reported.** Every small component of the national graph, apart from rail and one stray bus cluster, is a minor mode: light rail, cable tram and shared taxis share no `stop_id` with any bus stop, so in a stop-id-based model they cannot touch the bus network. Notebook 03's "a dozen components" is therefore mostly an encoding effect, and any multimodal transfer analysis (notebook 17) must work from spatial proximity rather than shared identifiers or it will conclude, wrongly, that no interchange exists.
* **Honest limits.** Weights are scheduled trips, never passengers - GTFS has no ridership, so "load" here means *supply*. Damage is measured within a single mode, so a light-rail failure that a parallel bus line would absorb still scores as a full disconnection. The feed is one snapshot: only the light-rail lines operating in it appear, and lines opened later do not exist in these numbers. Finally, betweenness is normalised within each mode graph, so a value of 0.2 in the Carmelit and a value of 0.2 nationally are not the same quantity and must not be compared across tables.